# Labo 33 · Carnet Colab — Plus vite et plus sobre

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "codecarbon"], check=False)
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = x_train[..., np.newaxis].astype("float32") / 255.0
x_test = x_test[..., np.newaxis].astype("float32") / 255.0

def cnn(normalisation=False, dropout=0.0):
    couches = [keras.Input(shape=(28, 28, 1))]
    for filtres in [32, 64]:
        couches.append(keras.layers.Conv2D(filtres, 3, padding="same"))
        if normalisation:
            couches.append(keras.layers.BatchNormalization())
        couches += [keras.layers.Activation("relu"), keras.layers.MaxPooling2D(2)]
    couches += [keras.layers.Flatten()]
    if dropout:
        couches.append(keras.layers.Dropout(dropout))
    couches.append(keras.layers.Dense(10, activation="softmax", dtype="float32"))   # sortie en float32, même en précision mixte
    modele = keras.Sequential(couches)
    modele.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modele
print(x_train.shape)

## 1. La taille des lots

**À toi** : mesure le temps d'**une** époque (`fit(..., epochs=1, verbose=0)` entre deux `time.time()`) avec des lots de 32, puis de 512. Fais d'abord une époque « à vide » pour que le GPU soit prêt.

In [ ]:
# À toi

## 2. La précision mixte

En précision mixte, les calculs se font en `float16`, deux fois plus légers; les poids restent en `float32`. Les GPU récents (comme le T4) ont des circuits spécialisés pour le `float16`. **À toi** : active la politique `"mixed_float16"`, refais la mesure avec des lots de 512, puis remets la politique `"float32"`.

In [ ]:
# À toi

## 3. Mesurer l'énergie avec CodeCarbon

`EmissionsTracker` mesure la consommation du processeur, du GPU et de la mémoire pendant un calcul, détecte la région à partir de l'adresse IP et estime les émissions. **À toi** : mesure un entraînement de 5 époques (lots de 128), puis affiche l'énergie en kWh et les émissions en kg éq. CO₂ (`tracker.final_emissions_data.energy_consumed` et `.emissions`).

In [ ]:
from codecarbon import EmissionsTracker
# À toi

**Question.** Colab tourne dans un centre de données de Google, rarement au Québec. Recalcule les émissions avec l'intensité du Québec (34.5 g/kWh, selon Hydro-Québec) à partir de l'énergie mesurée.

## 4. Normalisation par lots et dropout

**À toi** : entraîne 8 époques (lots de 128, `validation_split=0.1`) le CNN ordinaire, puis le CNN avec `normalisation=True` et `dropout=0.3`. Trace les deux courbes de perte de validation et compare les exactitudes au test.

In [ ]:
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : le temps par époque pour les lots de 32 et de 512, avec et sans précision mixte; l'énergie, les émissions et la région détectée par CodeCarbon, et ton recalcul pour le Québec; l'exactitude au test avec et sans normalisation par lots et dropout.